# Step 0: setup and data check

Gets the lab's code, prints the package versions, and checks the compendium. Run once, before the other notebooks.

Settings: where the project folder, the data and the lab's code are.

In [ ]:
import os, sys
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # hide TensorFlow's startup messages

PROJECT = os.path.expandvars("$SCRATCH/CSC-488-Code")    # project folder: data, lab code and results
DATA_PATH = os.path.join(PROJECT, "data", "Copy of ecoli_MG1655plus_genome_part2_log_counts_norm_01.csv")
SEQADAGE = os.path.join(PROJECT, "seqADAGE")              # the lab's code (my fork, Jordin branch)
RUN_DIR = os.path.join(PROJECT, "runs")                   # every notebook saves and reads its results here
os.chdir(PROJECT)

import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

Download the lab's seqADAGE code (my fork, `Jordin` branch) into the project folder, if it isn't there yet.

In [ ]:
if not os.path.isdir(SEQADAGE):
    subprocess.run(["git", "clone", "--branch", "Jordin", "https://github.com/palmerij-pixel/seqADAGE.git", SEQADAGE],
                   check=True)
print("lab code:", SEQADAGE)

Versions of the main packages (thesis Table 4).

In [ ]:
import scipy, statsmodels, matplotlib, plotnine, tensorflow, keras, keras_tuner, shap

print("python", sys.version.split()[0])
for m in [tensorflow, keras, keras_tuner, shap, np, scipy, pd, statsmodels, matplotlib, plotnine]:
    print(m.__name__, m.__version__)

Load the compendium (genes x samples) and check its size, value range, missing values and duplicate names.

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
vals = df.values
print("genes x samples:", df.shape)
print("value range:", float(vals.min()), "to", float(vals.max()))
print("missing values:", int(np.isnan(vals).sum()))
print("duplicate gene names:", int(df.index.duplicated().sum()),
      "| duplicate sample names:", int(df.columns.duplicated().sum()))
df.iloc[:5, :6]

Histogram of one million randomly chosen values.

In [ ]:
rng = np.random.default_rng(0)
picked = vals[rng.integers(0, vals.shape[0], 1_000_000), rng.integers(0, vals.shape[1], 1_000_000)]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.hist(picked, bins=50)
ax.set(xlabel="expression value (scaled 0 to 1)", ylabel="count")
plt.tight_layout()
os.makedirs(RUN_DIR, exist_ok=True)
fig.savefig(os.path.join(RUN_DIR, "data_check.png"), dpi=120)
plt.show()